# Identity Manipulation

For Identity Manipulation, the key idea is that money launderers often create multiple identities that appear separate but share hidden attributes:

- Same phone number
- Same address
- Same device
- Same IP address
- Same employer
- Same beneficial owner
- Similar names

This is fundamentally a graph problem, not a transaction problem.

This notebook focues on:

1. Create customer identities
2. Create hidden links
3. Build an identity graph
4. Compute graph features
5. Detect suspicious clusters
6. Train a risk model

In [2]:
#!pip install Faker

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 2.0/2.0 MB 28.2 MB/s  0:00:00



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
import numpy as np

import networkx as nx

from faker import Faker

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

import matplotlib.pyplot as plt

fake = Faker()

np.random.seed(42)

In [ ]:
# Generate customer dataset
n_customers = 2000

customers = []

for customer_id in range(n_customers):

    customers.append([
        customer_id,
        fake.name(),
        fake.address(),
        fake.phone_number(),
        fake.ipv4(),
        fake.company(),
        0
    ])

customers = pd.DataFrame(
    customers,
    columns=[
        "customer_id",
        "name",
        "address",
        "phone",
        "ip",
        "company",
        "suspicious"
    ]
)

customers.head()

In [ ]:
# Create syntehtic identity rings - groups of customers sharing information
# These represent:
# - mule rings
# - shell company groups
# - synthetic identities

for ring in range(50):

    members = np.random.choice(
        customers.index,
        size=np.random.randint(3,10),
        replace=False
    )

    shared_phone = fake.phone_number()
    shared_address = fake.address()

    for idx in members:

        customers.loc[idx, "phone"] = shared_phone
        customers.loc[idx, "address"] = shared_address
        customers.loc[idx, "suspicious"] = 1

In [ ]:
# Sanity check
customers["suspicious"].value_counts()

In [ ]:
# Build identity graph with nodes:
# - Customer
# - Phone
# - Address
# - IP
# - Company
G = nx.Graph()

In [ ]:
# Add customer nodes
for _, row in customers.iterrows():

    customer_node = f"C_{row.customer_id}"

    G.add_node(
        customer_node,
        node_type="customer"
    )

In [ ]:
# Add attribute nodes
for _, row in customers.iterrows():

    c = f"C_{row.customer_id}"

    phone = f"P_{row.phone}"
    address = f"A_{hash(row.address)}"
    ip = f"I_{row.ip}"
    company = f"B_{row.company}"

    G.add_edge(c, phone)
    G.add_edge(c, address)
    G.add_edge(c, ip)
    G.add_edge(c, company)

In [ ]:
# Graph summary
print(
    "Nodes:",
    G.number_of_nodes()
)

print(
    "Edges:",
    G.number_of_edges()
)

In [ ]:
# Visualize small example
sample_nodes = list(G.nodes())[:100]

subgraph = G.subgraph(sample_nodes)

plt.figure(figsize=(10,8))

nx.draw(
    subgraph,
    node_size=50,
    with_labels=False
)

plt.show()

In [ ]:
# Degree centrality - Shared identities create unusually connected nodes.
degree = nx.degree_centrality(G)

degree_df = pd.DataFrame(
    degree.items(),
    columns=["node","degree"]
)

degree_df.sort_values(
    "degree",
    ascending=False
).head()

In [ ]:
# Betweeness centrality - usefull for hidden connectors
betweenness = nx.betweenness_centrality(
    G,
    k=500
)

betweenness_df = pd.DataFrame(
    betweenness.items(),
    columns=[
        "node",
        "betweenness"
    ]
)

betweenness_df.sort_values(
    "betweenness",
    ascending=False
).head()

In [ ]:
# Detect communities
from networkx.algorithms.community import (
    greedy_modularity_communities
)

communities = list(
    greedy_modularity_communities(G)
)

len(communities)

In [ ]:
# Community sizes
community_sizes = [
    len(c)
    for c in communities
]

pd.Series(
    community_sizes
).describe()
# Large unusual communities often indicate:
# - identity farms
# - mule networks
# - shell structures

In [ ]:
# Create customer-level features
features = []

for customer_id in customers.customer_id:

    node = f"C_{customer_id}"

    features.append([
        customer_id,
        degree.get(node,0),
        betweenness.get(node,0)
    ])

features = pd.DataFrame(
    features,
    columns=[
        "customer_id",
        "degree",
        "betweenness"
    ]
)

In [ ]:
# Count shared attributes
shared_counts = []

for customer_id in customers.customer_id:

    node = f"C_{customer_id}"

    neighbors = list(
        G.neighbors(node)
    )

    shared = 0

    for n in neighbors:

        if G.degree(n) > 1:
            shared += 1

    shared_counts.append(shared)

features["shared_attributes"] = (
    shared_counts
)

In [ ]:
# Merge labels
features = features.merge(
    customers[
        [
            "customer_id",
            "suspicious"
        ]
    ],
    on="customer_id"
)

features.head()

In [ ]:
# Feature inspection
features.groupby(
    "suspicious"
).mean()
# You should observe suspicious identities having:
# - higher degree
# - more shared attributes

In [ ]:
# Train ML model
X = features[
    [
        "degree",
        "betweenness",
        "shared_attributes"
    ]
]

y = features["suspicious"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

model = RandomForestClassifier(
    n_estimators=200,
    max_depth=6,
    random_state=42
)

model.fit(
    X_train,
    y_train
)

In [ ]:
# Evaluate
preds = model.predict(X_test)

print(
    classification_report(
        y_test,
        preds
    )
)

In [ ]:
# Feature importance
importance = pd.Series(
    model.feature_importances_,
    index=X.columns
)

importance.sort_values().plot(
    kind="barh"
)

plt.show()

In [ ]:
# Risk scoring
features["risk_score"] = (
    model.predict_proba(X)
    [:,1]
)

features.sort_values(
    "risk_score",
    ascending=False
).head(20)

In [ ]:
# Explain top alert
top_customer = (
    features
    .sort_values(
        "risk_score",
        ascending=False
    )
    .iloc[0]
    .customer_id
)

node = f"C_{int(top_customer)}"

list(G.neighbors(node))

In [ ]:
# Node2Vec embeddings
from node2vec import Node2Vec

node2vec = Node2Vec(
    G,
    dimensions=64,
    walk_length=20,
    num_walks=100
)

model_embeddings = node2vec.fit()
# Each customer becomes a 64-dimensional vector capturing network behavior.
# These embeddings are often fed into:
# - XGBoost
# - LightGBM
# - Graph Neural Networks
# and typically outperform hand-crafted graph features.

Common systems flow:
```
Customer Graph
       ↓
GraphSAGE
       ↓
Customer Embedding
       ↓
Risk Classifier
       ↓
SAR Recommendation
```

This is very close to what large financial institutions and specialized AML vendors use today for detecting:
- synthetic identities
- mule networks
- shell company structures
- beneficial ownership concealment
- organized laundering rings.